In [1]:
import numpy as np
from scipy.spatial import cKDTree

In [2]:
lats = np.load(
    "/home/users/mendrika/Object-Based-LSTMConv/notebooks/model/deploy/geolocation/nrt_lats_africa.npy"
)
lons = np.load(
    "/home/users/mendrika/Object-Based-LSTMConv/notebooks/model/deploy/geolocation/nrt_lons_africa.npy"
)

assert lats.shape == lons.shape == (2015, 2186)

In [3]:
valid = (lats > -900) & (lons > -900)

lat_min = lats[valid].min()
lat_max = lats[valid].max()
lon_min = lons[valid].min()
lon_max = lons[valid].max()

In [4]:
print(lat_min, lat_max, lon_min, lon_max)

-39.90967309981029 26.56557458683449 -23.10216323592575 79.47847868562897


In [5]:
ny = nx = 512

lat_grid = np.linspace(lat_min, lat_max, ny)
lon_grid = np.linspace(lon_min, lon_max, nx)

lon2d, lat2d = np.meshgrid(lon_grid, lat_grid)

source_points = np.column_stack((lons[valid], lats[valid]))
target_points = np.column_stack((lon2d.ravel(), lat2d.ravel()))

tree = cKDTree(source_points)
dist, idx = tree.query(target_points)

np.save("NRT_msg_to_regular_idx.npy", idx)

In [9]:
import os 

dx = lon_grid[1] - lon_grid[0]
dy = lat_grid[1] - lat_grid[0]

np.savez(
    os.path.join("/home/users/mendrika/PANCAST/data/geolocation", "grid_regular_512.npz"),
    lon2d=lon2d,
    lat2d=lat2d,
    lon_grid=lon_grid,
    lat_grid=lat_grid,
    dx=dx,
    dy=dy,
    lon_min=lon_grid.min(),
    lon_max=lon_grid.max(),
    lat_min=lat_grid.min(),
    lat_max=lat_grid.max()
)